GD and Newton

imports

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


OUTPUT_DIR = Path("results_gd_newton")
FIGURE_DIR = Path("figures_gd_newton")
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

functions

In [ ]:
def quadratic(x):
    return float(x[0] ** 2 + x[1] ** 2)


def quadratic_gradient(x):
    return np.array([2.0 * x[0], 2.0 * x[1]], dtype=float)


def quadratic_hessian(x):
    return np.array([[2.0, 0.0], [0.0, 2.0]], dtype=float)


def rosenbrock(x):
    return float((1.0 - x[0]) ** 2 + 100.0 * (x[1] - x[0] ** 2) ** 2)


def rosenbrock_gradient(x):
    return np.array(
        [
            -2.0 * (1.0 - x[0]) - 400.0 * x[0] * (x[1] - x[0] ** 2),
            200.0 * (x[1] - x[0] ** 2),
        ],
        dtype=float,
    )


def rosenbrock_hessian(x):
    u, v = x[0], x[1]
    hxx = 2.0 - 400.0 * v + 1200.0 * u ** 2
    hxy = -400.0 * u
    return np.array([[hxx, hxy], [hxy, 200.0]], dtype=float)


def cosine_bumps(x):
    return float(
        x[0] ** 2 + x[1] ** 2 + 10.0 * np.cos(x[0]) + 10.0 * np.cos(x[1])
    )


def cosine_bumps_gradient(x):
    return np.array(
        [2.0 * x[0] - 10.0 * np.sin(x[0]), 2.0 * x[1] - 10.0 * np.sin(x[1])],
        dtype=float,
    )


def cosine_bumps_hessian(x):
    return np.array(
        [
            [2.0 - 10.0 * np.cos(x[0]), 0.0],
            [0.0, 2.0 - 10.0 * np.cos(x[1])],
        ],
        dtype=float,
    )


FUNCTIONS = {
    "quadratic": {
        "fun": quadratic,
        "grad": quadratic_gradient,
        "hess": quadratic_hessian,
        "xlim": (-3.5, 3.5),
        "ylim": (-3.5, 3.5),
    },
    "rosenbrock": {
        "fun": rosenbrock,
        "grad": rosenbrock_gradient,
        "hess": rosenbrock_hessian,
        "xlim": (-2.5, 2.5),
        "ylim": (-1.5, 3.5),
    },
    "cosine_bumps": {
        "fun": cosine_bumps,
        "grad": cosine_bumps_gradient,
        "hess": cosine_bumps_hessian,
        "xlim": (-6.5, 6.5),
        "ylim": (-6.5, 6.5),
    },
}

settings

In [ ]:
STARTS = {
    "start_1": (-2.0, 2.0),
    "start_2": (0.5, -1.5),
    "start_3": (3.0, 3.0),
}

GD_RATES = (0.001, 0.01, 0.1)
NEWTON_RATES = (0.1, 0.5, 1.0)
TOLERANCE = 1e-6
MAX_ITERATIONS = 2000
DIVERGE = 1e8

helper

In [ ]:
def _result(x, objective, trajectory, objective_history, iterations, elapsed, reason):
    return {
        "final_x": np.asarray(x, dtype=float).copy(),
        "final_value": float(objective(x)) if np.all(np.isfinite(x)) else np.nan,
        "iterations": int(iterations),
        "runtime_seconds": float(elapsed),
        "termination_reason": reason,
        "trajectory": np.asarray(trajectory, dtype=float),
        "objective_history": np.asarray(objective_history, dtype=float),
    }


def _ok(x, val):
    return np.all(np.isfinite(x)) and np.isfinite(val) and np.linalg.norm(x) < DIVERGE

landscapes

In [ ]:
grid = np.linspace(-4.0, 4.0, 140)
X, Y = np.meshgrid(grid, grid)

for function_name, spec in FUNCTIONS.items():
    figure, axis = plt.subplots(figsize=(6.8, 5.2))
    Z = np.array([[spec["fun"](np.array([x, y])) for x in grid] for y in grid])
    axis.contour(X, Y, Z, levels=22)
    axis.set_title("Landscape: " + function_name)
    axis.set_xlabel("x")
    axis.set_ylabel("y")
    figure.tight_layout()
    plt.show()
    plt.close(figure)

GD

In [ ]:
def gradient_descent(objective, gradient, hessian, start, learning_rate,
                     tolerance=TOLERANCE, max_iterations=MAX_ITERATIONS):
    x = np.asarray(start, dtype=float).copy()
    trajectory = [x.copy()]
    objective_history = [objective(x)]
    t0 = time.perf_counter()
    reason = "iteration_limit"
    iterations = 0

    for iteration in range(1, max_iterations + 1):
        grad = gradient(x)
        new_x = x - learning_rate * grad
        val = objective(new_x)
        iterations = iteration
        trajectory.append(new_x.copy())
        objective_history.append(val)
        if not _ok(new_x, val):
            x = new_x
            reason = "diverged"
            break
        if np.linalg.norm(new_x - x) < tolerance:
            x = new_x
            reason = "step_tolerance"
            break
        x = new_x

    return _result(x, objective, trajectory, objective_history, iterations,
                   time.perf_counter() - t0, reason)

Newton

In [ ]:
def newton(objective, gradient, hessian, start, learning_rate,
           tolerance=TOLERANCE, max_iterations=MAX_ITERATIONS):
    x = np.asarray(start, dtype=float).copy()
    trajectory = [x.copy()]
    objective_history = [objective(x)]
    t0 = time.perf_counter()
    reason = "iteration_limit"
    iterations = 0

    for iteration in range(1, max_iterations + 1):
        grad = gradient(x)
        H = hessian(x)
        eig = np.linalg.eigvalsh(H)
        damp = 0.0
        if eig.min() <= 1e-8:
            damp = max(1e-6, 1e-4 - eig.min())
        try:
            step = np.linalg.solve(H + damp * np.eye(2), grad)
        except np.linalg.LinAlgError:
            step = grad
        new_x = x - learning_rate * step
        val = objective(new_x)
        iterations = iteration
        trajectory.append(new_x.copy())
        objective_history.append(val)
        if not _ok(new_x, val):
            x = new_x
            reason = "diverged"
            break
        if np.linalg.norm(new_x - x) < tolerance:
            x = new_x
            reason = "step_tolerance"
            break
        x = new_x

    return _result(x, objective, trajectory, objective_history, iterations,
                   time.perf_counter() - t0, reason)

methods

In [ ]:
OPTIMIZERS = {
    "GD": gradient_descent,
    "Newton": newton,
}

RATES = {
    "GD": GD_RATES,
    "Newton": NEWTON_RATES,
}

plots

In [ ]:
def plot_run(function_name, optimizer_name, start, learning_rate):
    spec = FUNCTIONS[function_name]
    out = OPTIMIZERS[optimizer_name](
        spec["fun"], spec["grad"], spec["hess"], start, learning_rate
    )
    pts = out["trajectory"]
    print(optimizer_name, function_name, "a=", learning_rate)
    print(" ", out["termination_reason"], "iters", out["iterations"],
          "x", out["final_x"], "f", out["final_value"])

    xs = np.linspace(spec["xlim"][0], spec["xlim"][1], 200)
    ys = np.linspace(spec["ylim"][0], spec["ylim"][1], 200)
    X, Y = np.meshgrid(xs, ys)
    Z = np.zeros_like(X)
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Z[i, j] = spec["fun"](np.array([X[i, j], Y[i, j]]))

    plt.figure(figsize=(6, 5))
    plt.contour(X, Y, Z, levels=25)
    plt.plot(pts[:, 0], pts[:, 1], "r.-", label=optimizer_name)
    plt.plot(pts[0, 0], pts[0, 1], "go", label="start")
    plt.plot(pts[-1, 0], pts[-1, 1], "bs", label="end")
    plt.xlim(*spec["xlim"])
    plt.ylim(*spec["ylim"])
    plt.title(function_name + "  " + optimizer_name + "  a=" + str(learning_rate))
    plt.legend()
    plt.xlabel("x")
    plt.ylabel("y")
    plt.tight_layout()
    plt.show()


DEMOS = [
    ("quadratic", "Newton", (-2.0, 2.0), 1.0),
    ("quadratic", "GD", (-2.0, 2.0), 0.1),
    ("rosenbrock", "GD", (-2.0, 2.0), 0.1),
    ("rosenbrock", "Newton", (-2.0, 2.0), 1.0),
    ("cosine_bumps", "GD", (-2.0, 2.0), 0.1),
    ("cosine_bumps", "Newton", (0.5, -1.5), 1.0),
]

for name, opt, start, a in DEMOS:
    plot_run(name, opt, start, a)

run all starts / step sizes

In [ ]:
def run_experiments():
    rows = []
    config_order = 0
    for function_name, spec in FUNCTIONS.items():
        for start_name, start in STARTS.items():
            for optimizer_name, optimizer in OPTIMIZERS.items():
                for learning_rate in RATES[optimizer_name]:
                    config_order += 1
                    out = optimizer(
                        spec["fun"], spec["grad"], spec["hess"], start, learning_rate
                    )
                    rows.append({
                        "config_order": config_order,
                        "function": function_name,
                        "optimizer": optimizer_name,
                        "start_name": start_name,
                        "start_x": start[0],
                        "start_y": start[1],
                        "learning_rate": learning_rate,
                        "final_x": out["final_x"][0],
                        "final_y": out["final_x"][1],
                        "final_value": out["final_value"],
                        "iterations": out["iterations"],
                        "runtime_seconds": out["runtime_seconds"],
                        "termination_reason": out["termination_reason"],
                    })
    raw = pd.DataFrame(rows)
    raw.to_csv(OUTPUT_DIR / "gd_newton_raw.csv", index=False)
    return raw


raw_results = run_experiments()
print("rows:", len(raw_results))
display(raw_results)

summary (GD a=0.001, Newton a=1)

In [ ]:
show = raw_results[
    ((raw_results["optimizer"] == "GD") & (raw_results["learning_rate"] == 0.001))
    | ((raw_results["optimizer"] == "Newton") & (raw_results["learning_rate"] == 1.0))
]
summary = (
    show.groupby(["function", "optimizer", "learning_rate"], sort=False)
    .agg(
        iteration=("iterations", "mean"),
        time_ms=("runtime_seconds", lambda s: 1000 * s.mean()),
        final_f=("final_value", "mean"),
        reasons=("termination_reason", lambda s: ", ".join(sorted(set(s)))),
    )
    .reset_index()
)
display(summary)

trajectory plots

In [ ]:
def make_gd_newton_figures(raw):
    grid = np.linspace(-4.0, 4.0, 160)
    X, Y = np.meshgrid(grid, grid)
    for function_name, spec in FUNCTIONS.items():
        Z = np.array([[spec["fun"](np.array([x, y])) for x in grid] for y in grid])
        figure, axis = plt.subplots(figsize=(7.0, 5.6))
        axis.contour(X, Y, Z, levels=22)
        for start_name, start in STARTS.items():
            axis.scatter(*start, s=45, label=start_name.replace("start_", "start "))
        axis.set_title("Landscape: " + function_name)
        axis.set_xlabel("x")
        axis.set_ylabel("y")
        axis.legend(fontsize=8, loc="upper left")
        figure.tight_layout()
        figure.savefig(FIGURE_DIR / ("landscape_" + function_name + ".png"), dpi=160)
        plt.show()
        plt.close(figure)

    for function_name, spec in FUNCTIONS.items():
        figure, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))
        for axis, optimizer_name in zip(axes, OPTIMIZERS):
            a = 0.1 if optimizer_name == "GD" else 1.0
            start = STARTS["start_1"]
            if function_name == "cosine_bumps" and optimizer_name == "Newton":
                start = STARTS["start_2"]
            out = OPTIMIZERS[optimizer_name](
                spec["fun"], spec["grad"], spec["hess"], start, a
            )
            pts = out["trajectory"]
            xs = np.linspace(spec["xlim"][0], spec["xlim"][1], 150)
            ys = np.linspace(spec["ylim"][0], spec["ylim"][1], 150)
            XX, YY = np.meshgrid(xs, ys)
            Z = np.array([[spec["fun"](np.array([xx, yy])) for xx in xs] for yy in ys])
            axis.contour(XX, YY, Z, levels=18, colors="#aaaaaa", linewidths=0.65)
            axis.plot(pts[:, 0], pts[:, 1], color="tab:blue", linewidth=1.2)
            axis.scatter(pts[0, 0], pts[0, 1], c="tab:green", marker="o", label="start", zorder=4)
            axis.scatter(pts[-1, 0], pts[-1, 1], c="tab:red", marker="X", label="final", zorder=5)
            axis.set_title(optimizer_name + " | learning rate = " + str(a) + "\nstatus: " + out["termination_reason"])
            axis.set_xlabel("x")
            axis.set_ylabel("y")
            axis.legend(fontsize=8)
            axis.set_xlim(*spec["xlim"])
            axis.set_ylim(*spec["ylim"])
        figure.suptitle("Representative trajectories: " + function_name, fontsize=15)
        figure.tight_layout()
        figure.savefig(FIGURE_DIR / ("trajectories_" + function_name + ".png"), dpi=160)
        plt.show()
        plt.close(figure)

make_gd_newton_figures(raw_results)

complete results

In [ ]:
for fn in ["quadratic", "rosenbrock", "cosine_bumps"]:
    print("\n", fn)
    part = raw_results[raw_results["function"] == fn][
        [
            "optimizer",
            "start_name",
            "learning_rate",
            "iterations",
            "runtime_seconds",
            "final_value",
            "termination_reason",
        ]
    ].sort_values(["optimizer", "start_name", "learning_rate"])
    display(part)